# Потенциал, эквипотенциали и энтропия

**Главная мысль занятия.** Уравнение $E_x\,dx + E_y\,dy = 0$ говорит: при смещении вдоль кривой поле не совершает работы. Значит, его решения — эквипотенциали, а общий интеграл $\varphi(x, y) = C$ — сам потенциал. Если поле не потенциально, интегрирующий множитель меняет длину стрелок, не трогая кривых.

**Как запустить.** Нажмите **Ctrl+F9** (или «Среда выполнения» → «Выполнить всё»). На предупреждение «блокнот создан не Google» ответьте «Всё равно выполнить». Через несколько секунд ниже появятся картинки с кнопками и ползунками.

Всё управление — кнопками и ползунками.

In [ ]:
#@title Вычисления { display-mode: "form" }
# =============================================================
#  Движок: нить во внешнем поле, поле №199, идеальный газ.
#  Интерфейс — в следующих ячейках. Здесь ничего не рисуется.
# =============================================================
import numpy as np

C_EQ, C_LINE, C_SEP = '#2a78d6', '#eb6834', '#0b0b0b'
INK, INK2, GRID = '#0b0b0b', '#52514e', '#e6e5e0'


def fmt(x, d=2):
    return ('%.*f' % (d, x)).replace('.', ',').replace('-', '−')


# ---------- 1. нить во внешнем поле ----------
# k = q / (2 pi eps0): знак k — знак заряда; внешнее поле E0 вдоль +x.
# phi = -k ln r - E0 x,   поле E = -grad phi.
def thread_field(k, E0, x, y):
    with np.errstate(divide='ignore', invalid='ignore'):
        r2 = x * x + y * y
        return k * x / r2 + E0, k * y / r2


def thread_phi(k, E0, x, y):
    with np.errstate(divide='ignore', invalid='ignore'):
        return -0.5 * k * np.log(x * x + y * y) - E0 * x


def thread_lines(k, E0, x, y, upper):
    """Функция, постоянная вдоль силовых линий, отдельно в верхней и
    нижней полуплоскости (так она непрерывна: граница — ось x, а ось x
    сама целиком состоит из силовых линий)."""
    theta = np.arctan2(np.abs(y), x)
    if not upper:
        theta = -theta
    return k * theta + E0 * y


def zero_point(k, E0):
    """Точка, где поле равно нулю, или None."""
    if E0 <= 0:
        return None
    return (-k / E0, 0.0)


# ---------- 2. поле №199 ----------
# M dx + N dy = 0,  M = y^2,  N = -(x y + x^3);  mu = 1/x^3.
# После умножения на mu: -d( y^2/(2x^2) + y ) = 0.
def f199(x, y):
    return y * y, -(x * y + x ** 3)


def F199(x, y):
    return y * y / (2 * x * x) + y


_trapz = getattr(np, 'trapezoid', None) or np.trapz


def loop_work(field, xc, yc, w, h, mu=None, n=4000):
    """Работа поля (M, N) по прямоугольнику против часовой стрелки."""
    s = np.linspace(0.0, 1.0, n)
    corners = [(xc - w, yc - h), (xc + w, yc - h), (xc + w, yc + h), (xc - w, yc + h)]
    total = 0.0
    for i in range(4):
        (x0, y0), (x1, y1) = corners[i], corners[(i + 1) % 4]
        x, y = x0 + (x1 - x0) * s, y0 + (y1 - y0) * s
        M, N = field(x, y)
        if mu is not None:
            m = mu(x, y)
            M, N = M * m, N * m
        total += _trapz(M * (x1 - x0) + N * (y1 - y0), s)
    return total


# ---------- 3. идеальный газ, один моль ----------
R = 8.314


def gas(gamma):
    return dict(gamma=gamma, Cv=R / (gamma - 1))


def adiabat_T(g, V, V0, T0):
    """Адиабата через (V0, T0): T V^(gamma-1) = const."""
    return T0 * (V0 / V) ** (g['gamma'] - 1)


def path_isochore(g, V, T1, T2):
    return dict(V=np.full(2, V), T=np.array([T1, T2]),
                Q=g['Cv'] * (T2 - T1), S=g['Cv'] * np.log(T2 / T1))


def path_isotherm(g, T, V1, V2):
    return dict(V=np.array([V1, V2]), T=np.full(2, T),
                Q=R * T * np.log(V2 / V1), S=R * np.log(V2 / V1))


def path_adiabat(g, V1, T1, T2):
    V2 = V1 * (T1 / T2) ** (1 / (g['gamma'] - 1))
    V = np.geomspace(V1, V2, 60)
    return dict(V=V, T=adiabat_T(g, V, V1, T1), Q=0.0, S=0.0), V2


def three_paths(g, V1, T1, V2, T2):
    """Три пути из (V1, T1) в (V2, T2)."""
    a1, a2 = path_isochore(g, V1, T1, T2), path_isotherm(g, T2, V1, V2)
    b1, b2 = path_isotherm(g, T1, V1, V2), path_isochore(g, V2, T1, T2)
    c1, Vm = path_adiabat(g, V1, T1, T2)
    c2 = path_isotherm(g, T2, Vm, V2)
    return [('сначала нагрев, потом расширение', [a1, a2]),
            ('сначала расширение, потом нагрев', [b1, b2]),
            ('по адиабате, потом по изотерме', [c1, c2])]

---

## 1. Нить во внешнем поле: эквипотенциали и силовые линии

Синие линии — эквипотенциали $\varphi = C$, решения уравнения $E_x\,dx + E_y\,dy = 0$. Оранжевые — силовые линии: они идут вдоль поля, $E_y\,dx - E_x\,dy = 0$. Соседние силовые линии проведены так, что поток поля между ними одинаков.

In [ ]:
#@title Виджет: нить во внешнем поле { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

X_RNG, Y_RNG = (-3.2, 3.2), (-2.4, 2.4)
N_LINES = 16           # столько силовых линий выходит из заряда
K_ABS = 1.0            # |q| / (2 pi eps0) в условных единицах


class ThreadApp:

    def __init__(self):
        self._busy = False
        sl = dict(style={'description_width': '150px'}, layout=widgets.Layout(width='420px'))
        self.w_E0 = widgets.FloatSlider(value=0.6, min=0.0, max=2.0, step=0.05,
                                        description='внешнее поле E₀', **sl)
        self.w_q = widgets.ToggleButtons(options=[('заряд +', 1.0), ('заряд −', -1.0)],
                                         value=1.0)
        self.w_eq = widgets.Checkbox(value=True, indent=False, description='эквипотенциали')
        self.w_ln = widgets.Checkbox(value=True, indent=False, description='силовые линии')
        self.w_ar = widgets.Checkbox(value=False, indent=False, description='направление поля')
        b_only = widgets.Button(description='только заряд', layout=widgets.Layout(width='auto'))
        b_only.on_click(lambda _: self._set(w_E0=0.0))
        b_def = widgets.Button(description='как на паре', layout=widgets.Layout(width='auto'))
        b_def.on_click(lambda _: self._set(w_E0=0.6, w_q=1.0))
        self.out, self.info = widgets.Output(), widgets.HTML()
        for w in (self.w_E0, self.w_q, self.w_eq, self.w_ln, self.w_ar):
            w.observe(self._on_change, names='value')
        display(widgets.VBox([
            widgets.HTML("<h3 style='margin:2px 0'>Нить во внешнем однородном поле</h3>"),
            widgets.HBox([self.w_q, b_def, b_only]),
            self.w_E0,
            widgets.HBox([self.w_eq, self.w_ln, self.w_ar])]), self.out, self.info)
        x = np.linspace(*X_RNG, 401)
        self.yu = np.linspace(0.0, Y_RNG[1], 160)
        self.yl = np.linspace(Y_RNG[0], 0.0, 160)
        self.x = x
        self.X, self.Y = np.meshgrid(x, np.linspace(*Y_RNG, 321))
        self.Xu, self.Yu = np.meshgrid(x, self.yu)
        self.Xl, self.Yl = np.meshgrid(x, self.yl)
        self._refresh()

    def _draw(self):
        k, E0 = K_ABS * self.w_q.value, self.w_E0.value
        fig, ax = plt.subplots(figsize=(9.6, 7.0), dpi=88)
        fig.subplots_adjust(left=0.07, right=0.98, top=0.93, bottom=0.07)
        step = 2 * np.pi * K_ABS / N_LINES
        zp = zero_point(k, E0)
        inside = zp is not None and X_RNG[0] < zp[0] < X_RNG[1]
        msg = []

        if self.w_eq.value:
            phi = thread_phi(k, E0, self.X, self.Y)
            phi = np.ma.masked_where(self.X ** 2 + self.Y ** 2 < 0.05 ** 2, phi)
            base = thread_phi(k, E0, zp[0], 1e-12) if inside else 0.0
            lo, hi = np.percentile(phi.compressed(), [1, 99])
            levels = base + step * np.arange(np.floor((lo - base) / step),
                                             np.ceil((hi - base) / step) + 1)
            ax.contour(self.X, self.Y, phi, levels=levels, colors=C_EQ, linewidths=1.1,
                       linestyles='solid')
            if inside:
                ax.contour(self.X, self.Y, phi, levels=[base], colors=C_SEP,
                           linewidths=2.0, linestyles='solid')

        if self.w_ln.value:
            for Xh, Yh, up in ((self.Xu, self.Yu, True), (self.Xl, self.Yl, False)):
                psi = thread_lines(k, E0, Xh, Yh, up)
                lo, hi = psi.min(), psi.max()
                levels = step * np.arange(np.ceil(lo / step) - 1, np.floor(hi / step) + 2)
                ax.contour(Xh, Yh, psi, levels=levels, colors=C_LINE, linewidths=1.1,
                           linestyles='solid')
                if inside:
                    # Сетки yu/yl нарочно включают ось y = 0: пунктир идёт и по оси
                    # со стороны седла. Это настоящие усы сепаратрисы (при заряде «+»
                    # по оси в точку E = 0 входят две силовые линии), а не артефакт.
                    th = 0.0 if zp[0] > 0 else np.pi
                    sep = k * (th if up else -th)
                    ax.contour(Xh, Yh, psi, levels=[sep], colors=C_SEP,
                               linewidths=2.0, linestyles='--')
            ax.plot(X_RNG, [0, 0], color=C_LINE, lw=1.1)      # ось x — тоже силовые линии

        if self.w_ar.value:
            gx, gy = np.meshgrid(np.linspace(*X_RNG, 23), np.linspace(*Y_RNG, 17))
            ex, ey = thread_field(k, E0, gx, gy)
            n = np.hypot(ex, ey)
            ok = (gx ** 2 + gy ** 2) > 0.1
            ax.quiver(gx[ok], gy[ok], (ex / n)[ok], (ey / n)[ok], color='#9a9993',
                      scale=34, width=0.0025, headwidth=4, zorder=3)

        ax.plot(0, 0, 'o', ms=13, color='#e34948' if k > 0 else C_EQ,
                mec='white', mew=2, zorder=6)
        ax.text(0, 0, '+' if k > 0 else '−', color='white', ha='center', va='center',
                fontsize=11, fontweight='bold', zorder=7)
        if inside:
            ax.plot(*zp, 'o', ms=10, mfc='white', mec=C_SEP, mew=2, zorder=6)
            msg.append('Белый кружок — точка, где поле равно нулю. Чёрным выделены '
                       'эквипотенциаль (сплошная) и силовые линии (пунктир), которые '
                       'через неё проходят.')
        elif zp is not None:
            msg.append('Точка, где поле равно нулю, пока за краем картинки: увеличьте E₀, '
                       'и она придвинется к нити.')
        else:
            msg.append('Внешнего поля нет: только поле нити.')
        if E0 > 0:
            ax.annotate('', xy=(X_RNG[1] - 0.15, Y_RNG[1] - 0.25),
                        xytext=(X_RNG[1] - 0.95, Y_RNG[1] - 0.25),
                        arrowprops=dict(arrowstyle='->', color=INK2, lw=1.5))
            ax.text(X_RNG[1] - 0.55, Y_RNG[1] - 0.12, 'E₀', color=INK2, ha='center',
                    fontsize=11)
        ax.set_xlim(*X_RNG)
        ax.set_ylim(*Y_RNG)
        ax.set_aspect('equal')
        ax.set_xlabel('x')
        ax.set_ylabel('y')
        ax.set_title('синие — эквипотенциали,  оранжевые — силовые линии', loc='left',
                     color=INK, fontsize=11)
        for s in ('top', 'right'):
            ax.spines[s].set_visible(False)
        return fig, ' '.join(msg)

    def _refresh(self):
        with self.out:
            clear_output(wait=True)
            fig, msg = self._draw()
            plt.show()
        self.info.value = "<div style='font-size:14px'>%s</div>" % msg

    def _on_change(self, change=None):
        if not self._busy:
            self._refresh()

    def _set(self, **kw):
        self._busy = True
        try:
            for name, val in kw.items():
                getattr(self, name).value = val
        finally:
            self._busy = False
        self._refresh()


thread_app = ThreadApp()

### Что попробовать

1. **«только заряд».** Эквипотенциали — окружности, силовые линии — лучи. Это и есть теорема Гаусса в картинке: поток через любую окружность одинаков, поэтому все 16 лучей уходят на бесконечность.

2. **Плавно увеличивайте E₀.** Слева от заряда поле нити и внешнее поле направлены навстречу — появляется точка, где они гасят друг друга (белый кружок). Чем сильнее внешнее поле, тем ближе эта точка к нити. Посмотрите, как ведут себя чёрные линии около неё.

3. **Где силовые линии гуще?** Между соседними линиями поток одинаковый, поэтому там, где поле сильнее, линиям приходится идти теснее. Сравните окрестность заряда, точку $E = 0$ и края картинки.

4. **Включите «направление поля».** Стрелки везде перпендикулярны синим линиям и касаются оранжевых. Вдоль синей линии поле не совершает работы: это и есть уравнение $E_x\,dx + E_y\,dy = 0$.

5. **Переключите знак заряда.** Картинка отражается слева направо, точка $E = 0$ переходит на другую сторону. Почему именно туда?

---

## 2. Не потенциальное поле и интегрирующий множитель (№199)

Слева — «поле» $\vec E = (y^2,\ -xy - x^3)$ из №199, справа — то же поле, умноженное на $\mu = 1/x^3$. Чёрные кривые — решения уравнения $y^2\,dx - (xy + x^3)\,dy = 0$: на обеих картинках **одни и те же**. Стрелки (направление поля) на обеих картинках перпендикулярны кривым, а величина поля, показанная фоном, у двух полей разная. Фиолетовый прямоугольник — замкнутый путь, над картинками — работа поля по нему.

In [ ]:
#@title Виджет: интегрирующий множитель { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

X2, Y2 = (0.4, 2.8), (-1.9, 0.9)
W_LOOP, H_LOOP = 0.2, 0.28
MU = lambda x, y: 1.0 / x ** 3
C_LOOP = '#4a3aa7'


class MuApp:

    def __init__(self):
        sl = dict(style={'description_width': '110px'}, layout=widgets.Layout(width='380px'))
        self.w_xc = widgets.FloatSlider(value=1.3, min=X2[0] + W_LOOP + 0.02,
                                        max=X2[1] - W_LOOP - 0.02, step=0.02,
                                        description='путь: x центра', **sl)
        self.w_yc = widgets.FloatSlider(value=-0.8, min=Y2[0] + H_LOOP + 0.04,
                                        max=Y2[1] - H_LOOP - 0.04, step=0.02,
                                        description='путь: y центра', **sl)
        self.out = widgets.Output()
        for w in (self.w_xc, self.w_yc):
            w.observe(lambda ch: self._refresh(), names='value')
        display(widgets.HBox([self.w_xc, self.w_yc]), self.out)
        self.X, self.Y = np.meshgrid(np.linspace(*X2, 300), np.linspace(*Y2, 300))
        M, N = f199(self.X, self.Y)
        self.mag = [np.log10(np.hypot(M, N) + 1e-12),
                    np.log10(np.hypot(M, N) / self.X ** 3 + 1e-12)]
        self.F = F199(self.X, self.Y)
        self.gx, self.gy = np.meshgrid(np.linspace(X2[0] + 0.1, X2[1] - 0.1, 11),
                                       np.linspace(Y2[0] + 0.12, Y2[1] - 0.12, 15))
        gM, gN = f199(self.gx, self.gy)
        n = np.hypot(gM, gN)
        self.ux, self.uy = gM / n, gN / n          # у E и у μE направления одинаковые
        self._refresh()

    def _refresh(self):
        xc, yc = self.w_xc.value, self.w_yc.value
        A = [loop_work(f199, xc, yc, W_LOOP, H_LOOP),
             loop_work(f199, xc, yc, W_LOOP, H_LOOP, mu=MU)]
        titles = ['поле E', 'поле μE = E/x³']
        with self.out:
            clear_output(wait=True)
            fig, axs = plt.subplots(1, 2, figsize=(10.6, 5.2), dpi=88)
            fig.subplots_adjust(left=0.06, right=0.985, top=0.8, bottom=0.1, wspace=0.12)
            for i, ax in enumerate(axs):
                ax.pcolormesh(self.X, self.Y, self.mag[i], cmap='Greys', shading='auto',
                              vmin=self.mag[i].min() - 2.0, vmax=self.mag[i].max() + 1.5,
                              zorder=0)
                ax.contour(self.X, self.Y, self.F, levels=np.arange(-2.0, 4.01, 0.25),
                           colors=INK, linewidths=0.9, linestyles='solid', zorder=1)
                ax.quiver(self.gx, self.gy, self.ux, self.uy, color=C_LINE, angles='xy',
                          scale=26, width=0.005, pivot='middle', zorder=3)
                rx = [xc - W_LOOP, xc + W_LOOP, xc + W_LOOP, xc - W_LOOP, xc - W_LOOP]
                ry = [yc - H_LOOP, yc - H_LOOP, yc + H_LOOP, yc + H_LOOP, yc - H_LOOP]
                ax.plot(rx, ry, color=C_LOOP, lw=2.6, zorder=4)
                a = A[i] if abs(A[i]) > 5e-4 else 0.0
                ax.set_title('%s:  работа по пути = %s' % (titles[i], fmt(a, 3)),
                             loc='left', fontsize=11, color=INK)
                ax.set_xlim(*X2)
                ax.set_ylim(*Y2)
                ax.set_aspect('equal')
                ax.set_xlabel('x')
                for sp_ in ('top', 'right'):
                    ax.spines[sp_].set_visible(False)
            axs[0].set_ylabel('y')
            fig.text(0.06, 0.9, 'Стрелки показывают только направление — у двух полей оно '
                     'одинаковое.\nФон — величина поля: темнее — сильнее (каждое поле в своём '
                     'масштабе).', color=INK2, fontsize=10)
            plt.show()


mu_app = MuApp()

### Что попробовать

1. **Двигайте путь по картинке.** Слева работа по замкнутому пути почти всегда не ноль, и даже меняет знак — работа зависит от пути, потенциала нет. Справа она ноль, где бы ни стоял путь: у поля $\mu\vec E$ потенциал есть, это $\tilde\varphi = \dfrac{y^2}{2x^2} + y$, как на паре.

2. **Сравните фон.** Слева поле быстро растёт вправо — там работает $x^3$; справа оно поделено на $x^3$, и фон почти ровный. Стрелки при этом одни и те же: интегрирующий множитель ничего не поворачивает, он только перераспределяет величину поля так, чтобы работа по любому замкнутому пути исчезла.

3. **Почему кривые на картинках одинаковые?** Кривая — решение, если стрелки поля всюду ей перпендикулярны. Умножение стрелки на число этого не меняет.

Окно картинки начинается с $x = 0{,}4$: при $x = 0$ множитель $1/x^3$ бесконечен. Сама прямая $x = 0$ — тоже решение уравнения, и при умножении на $\mu$ оно теряется.

---

## 3. Теплота и энтропия: адиабаты — эквипотенциали энтропии

Один моль идеального газа на плоскости $(V, T)$. Теплота $\delta Q = C_V\,dT + \dfrac{RT}{V}\,dV$ — не полный дифференциал, поэтому полученная газом теплота зависит от пути. После умножения на $1/T$ получается $dS$, и $\int \delta Q/T$ от пути уже не зависит. Серые кривые — адиабаты, $\delta Q = 0$: линии постоянной энтропии.

In [ ]:
#@title Виджет: теплота и энтропия { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

V1, T1 = 1.0, 300.0
COLORS = ['#2a78d6', '#eb6834', '#1baf7a']


class GasApp:

    def __init__(self):
        sl = dict(style={'description_width': '140px'}, layout=widgets.Layout(width='400px'))
        self.w_V2 = widgets.FloatSlider(value=3.0, min=0.5, max=4.0, step=0.05,
                                        description='конечный объём, л', **sl)
        self.w_T2 = widgets.FloatSlider(value=450.0, min=150.0, max=900.0, step=1.0,
                                        description='конечная T, К', **sl)
        self.w_g = widgets.ToggleButtons(options=[('одноатомный, γ = 5/3', 5 / 3),
                                                  ('двухатомный, γ = 7/5', 7 / 5)],
                                         value=5 / 3)
        self.w_p = [widgets.Checkbox(value=True, indent=False, description=d,
                                     layout=widgets.Layout(width='auto'))
                    for d in ('путь 1', 'путь 2', 'путь 3')]
        self.out, self.info = widgets.Output(), widgets.HTML()
        for w in [self.w_V2, self.w_T2, self.w_g] + self.w_p:
            w.observe(lambda ch: self._refresh(), names='value')
        display(widgets.VBox([
            widgets.HTML("<h3 style='margin:2px 0'>Из (1 л, 300 К) в конечное состояние "
                         "тремя путями</h3>"),
            self.w_g, widgets.HBox([self.w_V2, self.w_T2]),
            widgets.HBox(self.w_p)]), self.out, self.info)
        self._refresh()

    def _refresh(self):
        g = gas(self.w_g.value)
        V2, T2 = self.w_V2.value, self.w_T2.value
        paths = three_paths(g, V1, T1, V2, T2)
        rows = []
        with self.out:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(9.6, 5.6), dpi=88)
            fig.subplots_adjust(left=0.08, right=0.84, top=0.92, bottom=0.11)
            V = np.geomspace(0.03, 8.0, 400)
            for T0 in np.geomspace(60, 3000, 16):
                ax.plot(V, adiabat_T(g, V, 1.0, T0), color='#c9c8c1', lw=1.0, zorder=1)
            vmin, vmax = 0.3, 4.5
            for i, (name, segs) in enumerate(paths):
                Q = sum(s['Q'] for s in segs)
                S = sum(s['S'] for s in segs)
                for s in segs:
                    vmin = min(vmin, s['V'].min())
                    vmax = max(vmax, s['V'].max())
                if self.w_p[i].value:
                    for j, s in enumerate(segs):
                        ax.plot(s['V'], s['T'], color=COLORS[i], lw=2.6,
                                ls=(0, (5, 4)) if i == 2 else '-', zorder=3 + i,
                                label='путь %d' % (i + 1) if j == 0 else None)
                rows.append((i + 1, name, Q, S))
            ax.plot([V1], [T1], 'o', ms=9, color=INK, zorder=5)
            ax.plot([V2], [T2], 'o', ms=9, color=INK, zorder=5)
            ax.text(V1, T1, '  начало', va='top', color=INK, fontsize=10)
            ax.text(V2, T2, '  конец', va='bottom', color=INK, fontsize=10)
            ax.set_xscale('log')
            ax.set_xlim(vmin * 0.8, vmax * 1.25)
            ax.set_ylim(100, 1000)
            ticks = [t for t in (0.1, 0.2, 0.3, 0.5, 1, 2, 3, 4, 6, 8)
                     if vmin * 0.8 <= t <= vmax * 1.25]
            ax.set_xticks(ticks)
            ax.set_xticklabels([fmt(t, 1).replace(',0', '') for t in ticks])
            ax.minorticks_off()
            ax.set_xlabel('V, л (логарифмическая шкала)')
            ax.set_ylabel('T, К')
            ax.set_title('серые — адиабаты (S = const)', loc='left', color=INK, fontsize=11)
            if any(w.value for w in self.w_p):
                ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1.0), frameon=False,
                          fontsize=10)
            ax.grid(color=GRID, lw=0.8)
            for s in ('top', 'right'):
                ax.spines[s].set_visible(False)
            plt.show()
        cells = ''.join(
            "<tr><td style='padding:2px 10px'><span style='color:%s'>■</span> путь %d</td>"
            "<td style='padding:2px 10px'>%s</td>"
            "<td style='padding:2px 10px; text-align:right'>%s Дж</td>"
            "<td style='padding:2px 10px; text-align:right'>%s Дж/К</td></tr>"
            % (COLORS[i - 1], i, name, fmt(Q, 0), fmt(S, 3)) for i, name, Q, S in rows)
        self.info.value = (
            "<table style='font-size:14px; border-collapse:collapse'>"
            "<tr><th></th><th style='text-align:left; padding:2px 10px'>как идём</th>"
            "<th style='padding:2px 10px'>теплота Q</th><th style='padding:2px 10px'>∫δQ/T</th>"
            "</tr>%s</table>" % cells)


gas_app = GasApp()

### Что попробовать

1. **Посмотрите на таблицу.** Теплота у трёх путей разная, а $\int \delta Q/T$ — одна и та же до последнего знака. Это и значит, что $\delta Q/T = dS$ — полный дифференциал.

2. **Путь 3** сначала идёт по адиабате — там теплота не подводится вовсе, — а потом по изотерме. На адиабате энтропия не меняется: адиабата — эквипотенциаль энтропии, как синие линии в первом виджете — эквипотенциали потенциала.

3. **Поставьте конечную точку на ту же адиабату, что и начало** — для одноатомного газа, например, 2 л и 189 К. Чему теперь равно $\int\delta Q/T$? А теплота по путям 1 и 2?

4. **Переключите газ на двухатомный.** Адиабаты становятся положе: $TV^{\gamma - 1} = \mathrm{const}$, а у двухатомного газа $\gamma - 1$ меньше. Почему это значит, что при сжатии двухатомный газ греется слабее?

---

## Вопросы к виджетам

1. **Почему синие и оранжевые линии в первом виджете всюду перпендикулярны?**

<details><summary>Ответ</summary>

Синие — кривые, вдоль которых $E_x\,dx + E_y\,dy = 0$, то есть смещение перпендикулярно полю. Оранжевые идут вдоль поля. Направление «вдоль поля» и направление «поперёк поля» перпендикулярны в каждой точке.

</details>

2. **Поставьте $E_0 = 2$ (правый край ползунка). Справа силовые линии, вышедшие из заряда, идут в полосе между чёрными пунктирами. Какой ширины эта полоса далеко справа?**

<details><summary>Ответ</summary>

Весь поток, вышедший из нити, по теореме Гаусса равен $q/\varepsilon_0$. Далеко справа поле почти однородное, $E_0$, и поток через полосу ширины $h$ равен $E_0 h$. Значит, $h = q/(\varepsilon_0 E_0)$. В единицах виджета $q/(2\pi\varepsilon_0) = 1$, так что $h = 2\pi/E_0 \approx 3{,}1$ при $E_0 = 2$; у правого края картинки полоса ещё у́же, около $2{,}7$, и расширяется до $3{,}1$ только вдали. Сильнее внешнее поле — у́же полоса; при $E_0 \to 0$ она становится бесконечно широкой, и линии от заряда расходятся во все стороны.

</details>

3. **Справа во втором виджете работа по замкнутому пути получается не ровно ноль, а, скажем, $0{,}000$ с крошечным остатком. Откуда остаток?**

<details><summary>Ответ</summary>

Работа считается численно: путь разбит на несколько тысяч кусочков, и сумма по ним приближает интеграл. Остаток — ошибка этого приближения, она убывает, если кусочков больше. Точное значение — ноль.

</details>

4. **В третьем виджете, путь 1: почему теплота на нём больше, чем на пути 2, если и начало, и конец одинаковые (при расширении с нагревом)?**

<details><summary>Ответ</summary>

На нагреве при постоянном объёме оба пути получают одинаково, $C_V(T_2 - T_1)$. Разница — на изотерме: путь 1 расширяется при высокой температуре $T_2$, путь 2 — при низкой $T_1$. Работа изотермического расширения $RT\ln(V_2/V_1)$ пропорциональна температуре, и столько же теплоты нужно подвести. При этом $\int\delta Q/T$ на изотерме равен $R\ln(V_2/V_1)$ при любой температуре — деление на $T$ ровно съедает разницу.

</details>

5. **Почему адиабаты в третьем виджете не пересекаются друг с другом?**

<details><summary>Ответ</summary>

Каждая адиабата — линия своего значения энтропии, $S = \mathrm{const}$. Если бы две адиабаты пересеклись, в точке пересечения у газа было бы два разных значения энтропии, а энтропия — функция состояния. По той же причине не пересекаются эквипотенциали в первом виджете (кроме точки, где поле равно нулю, — там у потенциала особая точка).

</details>